# Explainable Diabetes Risk Prediction — SHAP Analysis

This notebook implements comprehensive global and local explainability using **Tree SHAP** on the selected **XGBoost** model:
1. **Global Feature Importance:** Mean absolute SHAP values across the test cohort ($N = 154$)
2. **SHAP Beeswarm Summary:** Impact magnitude, distribution, and risk directionality
3. **Feature Dependence:** Non-linear effect curves for primary risk drivers (Glucose, BMI, Age, etc.)
4. **Individual Waterfall Explanations:** Patient-level additive attribution for high-risk and low-risk cases
5. **Native vs. SHAP Comparison:** Model gain ranking vs. marginal coalition ranking

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from explainability.shap_analysis import (
    load_best_model,
    get_tree_explainer,
    compute_shap_values,
    compute_global_importance,
    generate_shap_summary_bar,
    generate_shap_beeswarm,
    generate_dependence_plots,
    compare_feature_importance,
    explain_prediction
)
from ml.model_utils import load_processed_data, split_data, FEATURE_COLUMNS

print(f"SHAP version: {shap.__version__}")

## 1. Load Model & Prepare Evaluation Data

In [ ]:
model = load_best_model(PROJECT_ROOT / 'models' / 'best_model.pkl')
data = load_processed_data(PROJECT_ROOT / 'data' / 'diabetes.csv')
_, X_test, _, y_test = split_data(data, test_size=0.20, random_state=42)

print(f"Loaded model: {type(model).__name__}")
print(f"Evaluation samples: {len(X_test)}")

explainer = get_tree_explainer(model)
explanation = compute_shap_values(explainer, X_test)
print(f"Computed SHAP explanation: {explanation.shape}")

## 2. Global Feature Importance & Summary Plots

In [ ]:
df_importance = compute_global_importance(explanation, FEATURE_COLUMNS)
display(df_importance)

# Beeswarm Plot
plt.figure(figsize=(9, 6))
shap.plots.beeswarm(explanation, max_display=len(FEATURE_COLUMNS))
plt.show()

## 3. Feature Effect & Dependence Plots

In [ ]:
for feat in ['Glucose', 'BMI', 'Age']:
    plt.figure(figsize=(7, 5))
    shap.plots.scatter(explanation[:, feat], color=explanation)
    plt.title(f"SHAP Dependence: {feat}")
    plt.show()

## 4. Local Patient Explanations (Waterfall Plots)

In [ ]:
# Example 1: High risk patient (True positive)
plt.figure(figsize=(9, 6))
shap.plots.waterfall(explanation[0])
plt.title("Patient Case 1: High Diabetes Risk Explanation")
plt.show()

# Example 2: Low risk patient (True negative)
plt.figure(figsize=(9, 6))
shap.plots.waterfall(explanation[78])
plt.title("Patient Case 2: Low Diabetes Risk Explanation")
plt.show()

## 5. Native XGBoost Importance vs. SHAP Importance

In [ ]:
df_comparison = compare_feature_importance(
    model=model,
    df_shap_importance=df_importance,
    output_csv_path=PROJECT_ROOT / 'results' / 'feature_importance_comparison.csv',
    output_plot_path=PROJECT_ROOT / 'results' / 'feature_importance_comparison.png'
)
display(df_comparison)